# Nacimientos en Chile entre 2020 y 2023

## Sumativa 2 — Núcleo algorítmico, eficiencia y programación orientada a objetos

**MCDI500 · Grupo 2**  
**Integrantes:** Cristian Hurtado, Jorge Altamirano, Renzo Vílchez y Luis Paredes  
**Docente:** Dr. Omar Salinas Silva

Este cuaderno amplía el avance formativo de F3. Implementa cuatro maneras intercambiables de tratar las tallas ausentes, comprueba que reproducen F2 y mide cómo cambia el tiempo de dos algoritmos al aumentar el número de registros. La elección de método de talla sigue documentada en F2; esta implementación no cambia los CSV originales ni la pregunta del proyecto.
La organización del nuevo núcleo sigue la *Guía de apoyo de la Sumativa 2* y el *Apunte Fase 3* del curso (Universidad Andrés Bello, 2026a, 2026b).


## 1. Entorno y ruta de los datos

La primera celda detiene la ejecución si el kernel no coincide con las versiones de `requirements.txt`. Se usa el CSV preparado por F2 y se lee solo lo necesario. Las salidas se guardan como JSON pequeño en `F3/resultados/`; no se crea otro CSV de nacimientos.

In [1]:
# Comprobación previa: no continuar con un kernel distinto de requirements.txt.
from pathlib import Path as _Path
from importlib import metadata as _metadata
import sys as _sys

_inicio = _Path.cwd().resolve()
_raiz_entorno = next(
    (ruta for ruta in (_inicio, *_inicio.parents)
     if (ruta / "requirements.txt").is_file() and (ruta / "F1").is_dir() and (ruta / "F2").is_dir()),
    None,
)
if _raiz_entorno is None:
    raise RuntimeError("Abra el notebook dentro de la carpeta raíz del proyecto grupo 2.")
_fijadas = {}
for _linea in (_raiz_entorno / "requirements.txt").read_text(encoding="utf-8").splitlines():
    _linea = _linea.split("#", 1)[0].strip()
    if "==" in _linea:
        _paquete, _version = _linea.split("==", 1)
        _fijadas[_paquete.strip().lower()] = _version.strip()
_distintas = {}
for _paquete, _esperada in _fijadas.items():
    try:
        _actual = _metadata.version(_paquete)
    except _metadata.PackageNotFoundError:
        _actual = "no instalado"
    if _actual != _esperada:
        _distintas[_paquete] = f"{_actual} (requiere {_esperada})"
if _distintas:
    raise RuntimeError(
        "Kernel equivocado: las versiones no coinciden con requirements.txt. "
        f"Diferencias: {_distintas}. Intérprete actual: {_sys.executable}. "
        "En VS Code, arriba a la derecha: Select Kernel > Jupyter Kernels > "
        "Grupo 2 requirements (Python 3.13.5), o elija el Python de .venv. "
        "Seleccione ese entorno y vuelva a ejecutar desde la primera celda."
    )
print("Entorno correcto: versiones fijadas en requirements.txt verificadas.")

from pathlib import Path
import gc
import json
import platform
import sys
from datetime import date

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

def encontrar_raiz(inicio=Path.cwd()):
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'data/processed/nacimientos_preparados_propuesta.csv').is_file() and (candidato / 'src/f2_utilidades.py').is_file():
            return candidato
    raise FileNotFoundError('Abra el repositorio proyecto-grupo2-f3-cristian y compruebe la salida de F2.')

RAIZ = encontrar_raiz()
sys.dont_write_bytecode = True
sys.path.insert(0, str(RAIZ / 'F3/src'))
sys.path.insert(0, str(RAIZ / 'src'))
import f3_algoritmos as f3
import f2_utilidades as u
from f2_utilidades import aplanar

PREPARADO = RAIZ / 'data/processed/nacimientos_preparados_propuesta.csv'
ORIGINAL = RAIZ / 'data/raw/Serie_Nacimientos_2020_2023.csv'
RESULTADOS = RAIZ / 'F3/resultados'
RESULTADOS.mkdir(parents=True, exist_ok=True)
VERSIONES = {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__}
print('Fecha de ejecución:', date.today().isoformat())
print('Raíz:', RAIZ)
print('Versiones:', VERSIONES)

Entorno correcto: versiones fijadas en requirements.txt verificadas.


Fecha de ejecución: 2026-09-29
Raíz: D:\Magister UNAB\Curso 01\C01Week01\Evaluaciones\proyecto-grupo2-f3-cristian
Versiones: {'python': '3.13.5', 'pandas': '3.0.5', 'numpy': '2.5.2', 'scikit-learn': '1.9.0'}


## 2. Entrada preparada y decisiones de F2

F2 conserva 735.611 nacimientos y distingue la talla observada de la imputada. Aquí se leen región, glosa y talla observada para comprobar las cuatro alternativas ya comparadas: sin imputar, media general, mediana general y mediana regional con respaldo general. No se vuelven a limpiar los registros. La talla se mantiene en centímetros; no se escala porque esta comparación descriptiva no utiliza distancias. La codificación one-hot de región ya se muestra y verifica en `F3_Avance_Semana2.ipynb` como matriz separada de la versión legible.

In [2]:
COLUMNAS_TALLA = ['REGION_RESIDENCIA', 'GLOSA_REGION_RESIDENCIA', 'talla_observada_cm']
base = pd.read_csv(PREPARADO, usecols=COLUMNAS_TALLA)
assert len(base) == 735611
assert not base.REGION_RESIDENCIA.isna().any()
resumen_entrada = pd.DataFrame([{
    'nacimientos': len(base),
    'tallas_observadas': int(base.talla_observada_cm.notna().sum()),
    'tallas_ausentes_o_invalidas': int(base.talla_observada_cm.isna().sum()),
    'regiones': int(base.REGION_RESIDENCIA.nunique()),
}])
display(resumen_entrada)
display(base.head(4))

,nacimientos,tallas_observadas,tallas_ausentes_o_invalidas,regiones
0,735611,735014,597,16


,REGION_RESIDENCIA,GLOSA_REGION_RESIDENCIA,talla_observada_cm
0,13,Metropolitana de Santiago,47.0
1,13,Metropolitana de Santiago,47.0
2,13,Metropolitana de Santiago,44.0
3,13,Metropolitana de Santiago,46.0


### Comprobaciones de la salida de F2

F2 ya efectuó la limpieza y la preparación. Aquí se comprueba que su archivo de salida conserva el número de nacimientos, los años y meses previstos, la marca de madres menores de 20 años y las tallas observadas. También se compara la huella del CSV original con la registrada por F2. Estas verificaciones no sustituyen las pruebas detalladas de `F2/F2_Preparacion.ipynb`; muestran qué entrada recibe F3 y permiten detectar un archivo cambiado o incompleto. Los metadatos de F2 registran la **mediana regional aprobada por el grupo** para el tratamiento descriptivo de la talla.


In [3]:
metadatos_f2 = json.loads((RAIZ / 'docs/f2/metadatos.json').read_text(encoding='utf-8'))
columnas_control = ['ANO_NAC', 'MES_NAC', 'REGION_RESIDENCIA', 'madre_menor_20',
                    'talla_observada_cm', 'talla_analisis_cm', 'talla_imputada']
control_f2 = pd.read_csv(PREPARADO, usecols=columnas_control)
assert len(control_f2) == metadatos_f2['fuente']['filas'] == len(base)
assert set(control_f2.ANO_NAC.unique()) == {2020, 2021, 2022, 2023}
assert control_f2.MES_NAC.between(1, 12).all()
assert not control_f2.REGION_RESIDENCIA.isna().any()
assert pd.api.types.is_bool_dtype(control_f2.madre_menor_20)
assert not control_f2.madre_menor_20.isna().any()
observadas = control_f2.talla_observada_cm.notna()
assert control_f2.loc[observadas, 'talla_observada_cm'].gt(0).all()
assert np.allclose(control_f2.loc[observadas, 'talla_observada_cm'],
                   control_f2.loc[observadas, 'talla_analisis_cm'])
assert int(control_f2.talla_imputada.sum()) == metadatos_f2['preparacion']['imputados'] == 597
assert u.huella(ORIGINAL) == metadatos_f2['fuente']['sha256']
assert metadatos_f2['preparacion']['metodo_propuesto'] == 'mediana_region'
assert metadatos_f2['preparacion']['decision_grupo'] == 'mediana regional aprobada por el grupo'
evidencia_f2 = pd.DataFrame([
    ('Filas conservadas', len(control_f2), '735.611 nacidos vivos'),
    ('Años y meses válidos', '2020–2023; meses 1–12', 'categorías y calendario esperados'),
    ('Tallas observadas sin alterar', int(observadas.sum()), 'comparación con talla de análisis'),
    ('Tallas imputadas en la propuesta F2', int(control_f2.talla_imputada.sum()), 'igual al metadato F2'),
    ('Huella del CSV original', 'coincide', 'SHA-256 registrado en F2'),
], columns=['verificación', 'resultado', 'criterio'])
display(evidencia_f2)
print('Decisión del grupo registrada por F2:', metadatos_f2['preparacion']['decision_grupo'])
del control_f2


,verificación,resultado,criterio
0,Filas conservadas,735611,735.611 nacidos vivos
1,Años y meses válidos,2020–2023; meses 1–12,categorías y calendario esperados
2,Tallas observadas sin alterar,735014,comparación con talla de análisis
3,Tallas imputadas en la propuesta F2,597,igual al metadato F2
4,Huella del CSV original,coincide,SHA-256 registrado en F2


Decisión del grupo registrada por F2: mediana regional aprobada por el grupo


## 3. Cuatro clases para una misma tarea

El módulo `F3/src/f3_estrategias_talla.py` define una clase base con dos pasos: `ajustar` aprende los valores de referencia y `transformar` los aplica. Cuatro clases hijas implementan esos pasos de manera distinta. Esto es **herencia** (comparten una base) y **polimorfismo** (el comparador llama los mismos métodos sin preguntar qué clase recibió). Los valores aprendidos se guardan dentro de cada objeto y se consultan mediante una copia de `parametros`, evitando cambios accidentales desde fuera (**encapsulamiento**).

La clase abstracta define los métodos obligatorios según la documentación de Python (Python Software Foundation, s. f.-a). El diseño aplica el patrón *Strategy* a una decisión real del proyecto: cambiar el tratamiento de talla sin cambiar el comparador. Para reproducir la comparación descriptiva de F2, cada estrategia se ajusta sobre los mismos registros completos. Si en el futuro se construye un modelo predictivo, sus parámetros deberán aprenderse solo del conjunto de entrenamiento y luego aplicarse a prueba.

In [4]:
from f3_estrategias_talla import (
    EstrategiaTalla, SinImputar, MediaGeneral, MedianaGeneral,
    MedianaRegional, ComparadorImputacion,
)

estrategias = [SinImputar(), MediaGeneral(), MedianaGeneral(), MedianaRegional()]
assert all(isinstance(estrategia, EstrategiaTalla) for estrategia in estrategias)
comparador = ComparadorImputacion(estrategias)
escenarios_poo, comparacion_poo = comparador.comparar(base)
display(comparacion_poo.round(6))
print('Parámetros aprendidos por la estrategia regional:')
display(pd.DataFrame([
    {'region': region, 'mediana_cm': valor}
    for region, valor in estrategias[-1].parametros['medianas_por_region_cm'].items()
]).sort_values('region').reset_index(drop=True))

,metodo,n_validos,n_imputados,media_cm,mediana_cm,desv_cm
0,sin_imputar,735014,0,48.905500,49.0,2.402242
1,media_general,735611,597,48.905500,49.0,2.401267
2,mediana_general,735611,597,48.905576,49.0,2.401268
3,mediana_region,735611,597,48.905670,49.0,2.401291


Parámetros aprendidos por la estrategia regional:


,region,mediana_cm
0,1,49.0
1,2,49.0
2,3,49.0
3,4,49.0
4,5,49.0
5,6,49.0
6,7,49.0
7,8,50.0
8,9,49.0
9,10,49.0


## 4. Verificación frente a F2 y casos límite

Primero se exige igualdad **fila por fila** entre las cuatro clases nuevas y la función original de F2. Después se prueban casos pequeños que permiten comprobar resultados esperados y errores: ausencia en una región, región sin tallas válidas, datos sin faltantes, falta de una columna, valores inválidos y uso de una estrategia antes de ajustarla. Estas pruebas no reemplazan la revisión de los resultados completos, pero hacen visible el comportamiento ante entradas distintas.

In [5]:
escenarios_f2, comparacion_f2, _ = u.comparar_imputacion(base)
for nombre, anterior in escenarios_f2.items():
    pd.testing.assert_series_equal(escenarios_poo[nombre], anterior, check_names=False)
pd.testing.assert_frame_equal(
    comparacion_poo[['metodo', 'n_validos', 'n_imputados']].reset_index(drop=True),
    comparacion_f2[['metodo', 'n_validos', 'n_imputados']].reset_index(drop=True),
)
print('Equivalencia con F2: cuatro escenarios iguales en las 735.611 filas.')

mini = pd.DataFrame({
    'REGION_RESIDENCIA': [1, 1, 2, 2, 3],
    'talla_observada_cm': [48., np.nan, 50., np.nan, np.nan],
})
pruebas = []
regional = MedianaRegional().ajustar(mini)
assert regional.transformar(mini).tolist() == [48., 48., 50., 50., 49.]
pruebas.append(('Región sin tallas válidas', 'usa mediana general de respaldo'))
sin_faltantes = mini.iloc[[0, 2]].copy()
assert all(MedianaGeneral().ajustar(sin_faltantes).transformar(sin_faltantes) == sin_faltantes.talla_observada_cm)
pruebas.append(('Sin ausencias', 'conserva todas las tallas'))
prueba_nueva = pd.DataFrame({'REGION_RESIDENCIA': [17], 'talla_observada_cm': [np.nan]})
assert regional.transformar(prueba_nueva).iloc[0] == 49.
pruebas.append(('Región no observada al ajustar', 'usa mediana general de respaldo'))
copia_parametros = regional.parametros
copia_parametros['medianas_por_region_cm'][1] = -999.
assert regional.parametros['medianas_por_region_cm'][1] == 48.
pruebas.append(('Copia externa de parámetros', 'no modifica el valor guardado en la clase'))

def exigir_error(tipo, funcion):
    try:
        funcion()
    except tipo:
        return True
    raise AssertionError(f'Se esperaba {tipo.__name__}')

assert exigir_error(RuntimeError, lambda: MediaGeneral().transformar(mini))
pruebas.append(('Transformar antes de ajustar', 'error controlado'))
assert exigir_error(ValueError, lambda: MediaGeneral().ajustar(mini.drop(columns='talla_observada_cm')))
pruebas.append(('Falta una columna', 'error controlado'))
assert exigir_error(ValueError, lambda: MediaGeneral().ajustar(mini.assign(talla_observada_cm=np.nan)))
pruebas.append(('Todas las tallas ausentes', 'error controlado'))
assert exigir_error(ValueError, lambda: MediaGeneral().ajustar(mini.assign(talla_observada_cm=-1.)))
pruebas.append(('Talla observada negativa', 'error controlado'))
display(pd.DataFrame(pruebas, columns=['caso', 'resultado comprobado']))

Equivalencia con F2: cuatro escenarios iguales en las 735.611 filas.


,caso,resultado comprobado
0,Región sin tallas válidas,usa mediana general de respaldo
1,Sin ausencias,conserva todas las tallas
2,Región no observada al ajustar,usa mediana general de respaldo
3,Copia externa de parámetros,no modifica el valor guardado en la clase
4,Transformar antes de ajustar,error controlado
5,Falta una columna,error controlado
6,Todas las tallas ausentes,error controlado
7,Talla observada negativa,error controlado


### Codificación nominal: muestra de la matriz one-hot

El avance formativo de F3 ya aplicó one-hot a la región. Se reproduce aquí una muestra para que el evaluador vea la transformación junto al núcleo algorítmico. Se divide el conjunto con semilla fija; el codificador aprende las categorías solo en entrenamiento y después transforma prueba. La región legible sigue en el CSV preparado. Si surge una categoría nueva, no se agrega una columna: la fila codificada queda en ceros y se conserva el valor original para interpretarla (scikit-learn developers, s. f.; Pedregosa et al., 2011).


In [6]:
regiones_texto = base[['REGION_RESIDENCIA']].astype(str)
indices_entrenamiento, indices_prueba = train_test_split(
    np.arange(len(base)), test_size=0.20, random_state=42,
    stratify=regiones_texto.REGION_RESIDENCIA,
)
codificador = OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.uint8)
matriz_entrenamiento = codificador.fit_transform(regiones_texto.iloc[indices_entrenamiento])
matriz_prueba = codificador.transform(regiones_texto.iloc[indices_prueba])
columnas_onehot = codificador.get_feature_names_out(['REGION_RESIDENCIA']).tolist()
assert matriz_entrenamiento.shape[1] == matriz_prueba.shape[1] == len(columnas_onehot) == 16
assert np.asarray(matriz_prueba.sum(axis=1)).ravel().min() == 1
categoria_nueva = codificador.transform(pd.DataFrame({'REGION_RESIDENCIA': ['99']}))
assert categoria_nueva.shape[1] == 16 and categoria_nueva.nnz == 0
ejemplos = [int(np.flatnonzero(regiones_texto.REGION_RESIDENCIA.to_numpy() == r)[0])
            for r in ('1', '5', '13', '16')]
matriz_ejemplo = codificador.transform(regiones_texto.iloc[ejemplos]).toarray()
tabla_onehot = pd.DataFrame(matriz_ejemplo, columns=columnas_onehot,
                            index=[f'fila {i}: región {regiones_texto.iloc[i, 0]}' for i in ejemplos])
display(tabla_onehot)
print('Entrenamiento:', matriz_entrenamiento.shape, '| Prueba:', matriz_prueba.shape)
print('Categoría nueva: 16 columnas iguales y ningún 1; conservar la región original.')


,REGION_RESIDENCIA_1,REGION_RESIDENCIA_10,REGION_RESIDENCIA_11,REGION_RESIDENCIA_12,REGION_RESIDENCIA_13,REGION_RESIDENCIA_14,REGION_RESIDENCIA_15,REGION_RESIDENCIA_16,REGION_RESIDENCIA_2,REGION_RESIDENCIA_3,REGION_RESIDENCIA_4,REGION_RESIDENCIA_5,REGION_RESIDENCIA_6,REGION_RESIDENCIA_7,REGION_RESIDENCIA_8,REGION_RESIDENCIA_9
fila 4: región 1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
fila 21: región 5,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
fila 0: región 13,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
fila 183: región 16,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


Entrenamiento: (588488, 16) | Prueba: (147123, 16)
Categoría nueva: 16 columnas iguales y ningún 1; conservar la región original.


## 5. Eficiencia con distintos tamaños

La clase `AnalizadorNacimientos`, propuesta por Jorge Altamirano, comprueba el indicador sobre el conjunto preparado completo. Para medir 10.000, 100.000 y todos los registros se conservan las funciones independientes: así cada tamaño usa una muestra explícita. Se comparan un bucle de Python y `groupby` en esos tres tamaños. Antes de medir, se exige que los resultados sean iguales. Para cada método y tamaño se realizan tres repeticiones de tiempo con `time.perf_counter()` y se informa el menor tiempo. La memoria se observa en una ejecución **separada** mediante `tracemalloc`: su pico registra asignaciones rastreadas por Python, no toda la memoria nativa de pandas ni el máximo del proceso. También se registra el tamaño del `DataFrame` de salida; ninguno de estos valores reemplaza una medición integral del proceso.

Al crecer las filas, ambos métodos hacen trabajo proporcional al volumen de datos en este experimento. El bucle mantiene conteos por región; `groupby` puede crear estructuras intermedias. Las cifras muestran el costo observado en este equipo, no prueban una complejidad exacta para toda implementación de pandas. Se elige `groupby` si conserva el resultado y mantiene la ventaja temporal medida, considerando también los valores de memoria (Python Software Foundation, s. f.-b; The pandas development team, s. f.).


In [7]:
analizador_regional = f3.AnalizadorNacimientos(PREPARADO)
try:
    analizador_regional.proporcion_bucle()
except ValueError as error:
    assert 'cargar' in str(error)
else:
    raise AssertionError('El analizador debe exigir cargar los datos primero.')
datos_indicador = analizador_regional.cargar_preparado()
pd.testing.assert_frame_equal(
    analizador_regional.proporcion_bucle(), f3.proporcion_bucle(datos_indicador),
    check_dtype=False, rtol=1e-12,
)
pd.testing.assert_frame_equal(
    analizador_regional.proporcion_groupby(), f3.proporcion_groupby(datos_indicador),
    check_dtype=False, rtol=1e-12,
)
print('Clase regional y funciones independientes: resultados equivalentes.')
filas_medidas = []
for cantidad in (10000, 100000, len(datos_indicador)):
    muestra = datos_indicador.iloc[:cantidad]
    por_bucle = f3.proporcion_bucle(muestra)
    por_groupby = f3.proporcion_groupby(muestra)
    pd.testing.assert_frame_equal(por_bucle, por_groupby, check_dtype=False, rtol=1e-12)
    for metodo, funcion in (
        ('bucle', f3.proporcion_bucle),
        ('groupby', f3.proporcion_groupby),
    ):
        repeticiones = f3.medir_tiempos(lambda f=funcion, m=muestra: f(m), repeticiones=3)
        salida_memoria, pico_bytes = f3.medir_memoria_pico(lambda f=funcion, m=muestra: f(m))
        pd.testing.assert_frame_equal(salida_memoria, por_bucle, check_dtype=False, rtol=1e-12)
        filas_medidas.append({
            'filas': cantidad,
            'metodo': metodo,
            'tiempo_min_s': min(repeticiones),
            'tiempos_s': repeticiones,
            'memoria_entrada_mb': muestra.memory_usage(index=True, deep=True).sum() / 2**20,
            'memoria_salida_mb': salida_memoria.memory_usage(index=True, deep=True).sum() / 2**20,
            'pico_python_mb': pico_bytes / 2**20,
        })
tabla_eficiencia = pd.DataFrame(filas_medidas)
display(tabla_eficiencia[['filas', 'metodo', 'tiempo_min_s', 'memoria_entrada_mb', 'memoria_salida_mb', 'pico_python_mb']].round(4))
comparacion_tiempos = tabla_eficiencia.pivot(index='filas', columns='metodo', values='tiempo_min_s')
comparacion_tiempos['razon_bucle_groupby'] = comparacion_tiempos['bucle'] / comparacion_tiempos['groupby']
display(comparacion_tiempos.round(2))
print('Equivalencia del indicador: comprobada en los tres tamaños.')

Clase regional y funciones independientes: resultados equivalentes.


,filas,metodo,tiempo_min_s,memoria_entrada_mb,memoria_salida_mb,pico_python_mb
0,10000,bucle,0.0027,0.0860,0.0006,0.0162
1,10000,groupby,0.0008,0.0860,0.0006,0.3387
2,100000,bucle,0.0205,0.8584,0.0006,0.1576
3,100000,groupby,0.0018,0.8584,0.0006,2.7886
4,735611,bucle,0.1420,6.3139,0.0006,0.0168
5,735611,groupby,0.0092,6.3139,0.0006,21.7480


metodo,bucle,groupby,razon_bucle_groupby
filas,,,
10000,0.00,0.00,3.20
100000,0.02,0.00,11.50
735611,0.14,0.01,15.45


Equivalencia del indicador: comprobada en los tres tamaños.


## 6. Recursividad y registro de resultados

La función recursiva `aplanar` de F2 se usa sobre metadatos anidados de esta ejecución. No se emplea recursión para recorrer nacimientos: allí el bucle o la agrupación son más claros. Se conserva un JSON pequeño con versiones, comprobaciones de F2, vocabulario one-hot, parámetros aprendidos y mediciones de tiempo y memoria. Los parámetros permiten repetir cada transformación sin adivinar su valor; el conjunto preparado con región y centímetros permanece como versión legible.

In [8]:
resumen_sumativa = {
    'fecha': date.today().isoformat(),
    'versiones': VERSIONES,
    'entrada': {'filas': len(base), 'tallas_ausentes': int(base.talla_observada_cm.isna().sum())},
    'verificaciones': {
        'cuatro_escenarios_iguales_a_f2': True,
        'casos_limite_y_error': len(pruebas),
        'indicador_igual_en_tres_tamanos': True,
    },
    'estrategias': {e.nombre: e.parametros for e in estrategias},
    'evidencia_f2': {'validaciones': len(evidencia_f2), 'sha256_original': metadatos_f2['fuente']['sha256'],
                    'decision_grupo': metadatos_f2['preparacion']['decision_grupo']},
    'codificacion': {'categorias_entrenamiento': codificador.categories_[0].tolist(),
                     'columnas': columnas_onehot, 'filas_entrenamiento': len(indices_entrenamiento),
                     'filas_prueba': len(indices_prueba), 'misma_estructura': True},
    'benchmark': tabla_eficiencia.to_dict(orient='records'),
}
plano = aplanar(resumen_sumativa)
assert plano['verificaciones.cuatro_escenarios_iguales_a_f2'] is True
assert plano['entrada.filas'] == len(base)
salida_json = RESULTADOS / 'sumativa2_verificacion.json'
salida_json.write_text(json.dumps(resumen_sumativa, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(list(plano.items()), columns=['ruta', 'valor']).head(12))
print('Resultados guardados en:', salida_json.relative_to(RAIZ))

,ruta,valor
0,fecha,2026-09-29
1,versiones.python,3.13.5
2,versiones.pandas,3.0.5
3,versiones.numpy,2.5.2
4,versiones.scikit-learn,1.9.0
5,entrada.filas,735611
6,entrada.tallas_ausentes,597
7,verificaciones.cuatro_escenarios_iguales_a_f2,True
8,verificaciones.casos_limite_y_error,8
9,verificaciones.indicador_igual_en_tres_tamanos,True


Resultados guardados en: F3\resultados\sumativa2_verificacion.json


## 7. Interpretación y alcance

Las cuatro estrategias de talla reproducen exactamente los escenarios de F2. La clase regional propuesta por Jorge Altamirano reproduce los indicadores de las funciones independientes. Así, la organización orientada a objetos no cambia los datos ni los resultados. El grupo aprobó la mediana regional para el tratamiento descriptivo de la talla en F2; comparar estrategias no demuestra cuál recupera el valor real de una talla faltante. Las pruebas verifican casos normales, límite y errores. Las tablas de tiempo y memoria permiten elegir una implementación del indicador para este equipo y este volumen, pero no afirma que la misma razón de velocidad sea universal. La versión legible del dataset y la matriz one-hot del avance formativo permanecen separadas.

La ampliación se limita a F3: F1 y F2 siguen como antecedentes verificables. Para un futuro modelo predictivo, las estrategias con parámetros aprendidos deberían ajustarse solo con entrenamiento y aplicarse después a prueba. En este análisis descriptivo se ajustaron sobre todos los registros para comprobar los mismos cuatro resultados ya publicados en F2.

## Referencias

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://www.jmlr.org/papers/v12/pedregosa11a.html

Python Software Foundation. (s. f.-a). *abc — Abstract Base Classes*. https://docs.python.org/3/library/abc.html

Python Software Foundation. (s. f.-b). *time — Time access and conversions*. https://docs.python.org/3/library/time.html

scikit-learn developers. (s. f.). *OneHotEncoder*. https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

The pandas development team. (s. f.). *pandas.DataFrame.groupby*. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html

Thimbleby, H. (2024). Improving science that uses code. *The Computer Journal, 67*(4), 1381–1404. https://doi.org/10.1093/comjnl/bxad067

Universidad Andrés Bello. (2026a). *Guía de apoyo — Evaluación Sumativa 2 (Fase 3): Núcleo algorítmico, eficiencia e implementación orientada a objetos* [Material de curso]. MCDI500.

Universidad Andrés Bello. (2026b). *MCDI500: Apunte Fase 3* [Material de curso]. MCDI500.
